1.1装环境

In [3]:
import torch
from torch import nn,optim,tensor
from torch.optim import Adam
from torch.nn import functional as F
import math
import spacy
import time
import torchtext

In [4]:
print(f"PyTorch 版本: {torch.__version__}")
print(f"Torchtext 版本: {torchtext.__version__}")
print(f"SpaCy 语言包: {'已加载' if spacy.load('en_core_web_sm') else '失败'}")

PyTorch 版本: 2.1.0+cu121
Torchtext 版本: 0.16.0+cpu
SpaCy 语言包: 已加载


1.2configure配置参数

In [5]:
#GPU device setting
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

#模型参数
batch_size = 128 
max_len = 256 #单句最大长度

d_model = 512 #词嵌入向量维度
n_layers = 6 # encoder decoder层数量
n_heads = 8 #注意力头数 
ffn_hidden = 2048 #前向传播维度
drop_prob = 0.1
n_hidden = ffn_hidden

#optimizer parameter setting

init_lr = 1e-5 #初始学习率
factor = 0.9 #衰减因子，触发patience后，学习率衰减
adam_eps = 5e-9
patience = 10 #在验证集连续patience个epoch模型的表现是否提升
warmup = 100 #刚开始的warmup步数，从0到init_lr线性增加
epoch = 100 #训练总轮数
clip = 1.0 #梯度剪裁阈值 防止梯度爆炸
weight_decay = 5e-4 #权重衰减，L2正则化系数，防止过拟合
inf = float('inf')


1.3英德文tokenizer

In [ ]:
class Tokenizer:
    def __init__(self):
        """
        spacy为工业界常用的NLP库
        调用spacy.load()加载预训练小语言模型
        """
        self.spacy_de = spacy.load('de_core_news_sm')
        self.spacy_en = spacy.load('en_core_web_sm')
    
    def tokenize_de(self , text):
        return [tok.text for tok in self.spacy_de.tokenizer(text)]
    
    def tokenize_en(self , text):
        return [tok.text for tok in self.spacy_en.tokenizer(text)]
    
tokenizer = Tokenizer()
example = "Hello, how are you?"
tokens = tokenizer.tokenize_en(example)

print(f"原始文本: {example}")
print(f"分词结果: {tokens}")

原始文本: Hello, how are you?
分词结果: ['Hello', ',', 'how', 'are', 'you', '?']


In [7]:
example = 'two young , white males are outside near many bushes'
tokens = tokenizer.tokenize_en(example)
print(f"原始文本: {example}")
print(f"分词结果: {tokens}")

原始文本: two young , white males are outside near many bushes
分词结果: ['two', 'young', ',', 'white', 'males', 'are', 'outside', 'near', 'many', 'bushes']


1.4 dataloader 创建

In [ ]:
from torchtext.data import Field , BucketIterator
from torchtext.datasets.translation import Multi30k

class DataLoader :
    source: Field = None
    target: Field = None

    def __init__(self ,ext , tokenize_en ,tokenize_de ,init_token='<sos>', eos_token='<eos>', pad_token='<pad>', unk_token='<unk>'):
        self.ext = ext
        self.tokenize_en = tokenize_en
        self.tokenize_de = tokenize_de
        self.init_token = init_token #起始符
        self.eos_token = eos_token #终止符
        self.pad_token = pad_token
        self.unk_token = unk_token

        self.UNK_IDX, self.PAD_IDX, self.SOS_IDX, self.EOS_IDX = 0, 1, 2, 3
        self.specials = [self.unk_token, self.pad_token, self.init_token, self.eos_token]

        self.vocab_src = None
        self.vocab_trg = None

        print('dataset initializing start')

    def make_dataset(self):
        """
        定义处理规则与加载数据
        """
        if self.ext == ('.de' , '.en'):
            self.source = Field(tokenize = self.tokenize_de , init_token =self.init_token, 
                                eos_token = self.eos_token , lower = True , batch_first = True)
            self.target = Field(tokenize = self.tokenize_en , init_token =self.init_token,
                                eos_token = self.eos_token , lower = True , batch_first = True)
        elif self.ext == ('.en' , '.de'):
            self.source = Field(tokenize = self.tokenize_en , init_token =self.init_token, 
                                eos_token = self.eos_token , lower = True , batch_first = True)
            self.target = Field(tokenize = self.tokenize_de , init_token =self.init_token,
                                eos_token = self.eos_token , lower = True , batch_first = True)
        
        train_data , valid_data ,test_data = Multi30k.splits(exts = self.ext , fields = (self.source , self.target))
        return train_data , valid_data , test_data
    
    # --- 1. 构建词汇表 (取代之前的 Field.build_vocab) ---
    def yield_tokens(self , data_iter , language):
        """生成器：遍历数据集，生成单词序列供构建词表使用"""
        for src_text , trg_text in data_iter:
            if language == 'src':
                yield self.tokenize_de(src_text.lower())
            else:
                yield self.tokenize_en(trg_text.lower())

    def make_iter(self , train , validate , test , batch_size , device):
        """
        创建迭代器， BucketTIterator将长度相近的句子自动分到同一个Batch里
        """

        train_iterator , valid_iterator , test_iterator = BucketIterator.splits((train , validate ,test),batch_size = batch_size , device = device)
        print('dataset initializin done')
        return train_iterator , valid_iterator , test_iterator


loader = DataLoader(ext = ('.en','.de'),
                    tokenize_en = tokenizer.tokenize_en,
                    tokenize_de = tokenizer.tokenize_de,
                    init_token = '<sos>',
                    eos_token = '<eos>'
                    )
print('\n--------0. 根据spacy mutli30k 创建数据集-------')
train, valid, test = loader.make_dataset()
print(train.examples[0].src)
print(train.examples[0].trg)
print(len(train.examples))
print(len(test.examples))
print(len(valid.examples))

ImportError: cannot import name 'Field' from 'torchtext.data' (/home/wangxinqi_26574/anaconda3/envs/mytransformer/lib/python3.10/site-packages/torchtext/data/__init__.py)